In [ ]:
import time
from datetime import datetime, timezone

from pyspark.sql import functions as F

from config import landing_incremental_dir, silver_table

dbutils.widgets.text("batch_size", "100")
dbutils.widgets.text("num_batches", "3")
dbutils.widgets.text("update_ratio", "0.3")
dbutils.widgets.text("invalid_ratio", "0.02")
dbutils.widgets.text("interval_seconds", "5")

BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
NUM_BATCHES = int(dbutils.widgets.get("num_batches"))
UPDATE_RATIO = float(dbutils.widgets.get("update_ratio"))
INVALID_RATIO = float(dbutils.widgets.get("invalid_ratio"))
INTERVAL_SECONDS = int(dbutils.widgets.get("interval_seconds"))

In [ ]:
review_columns = ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool", "text", "date"]

source_df = spark.table(silver_table("review")).select(*review_columns)
total_reviews = source_df.count()
sample_fraction = min(1.0, (BATCH_SIZE * 3) / total_reviews)

for batch in range(1, NUM_BATCHES + 1):
    sampled_df = (
        source_df
        .sample(fraction=sample_fraction)
        .orderBy(F.rand())
        .limit(BATCH_SIZE)
        .withColumn("_roll", F.rand())
    )
    
    updates_df = (
        sampled_df.filter(F.col("_roll") < UPDATE_RATIO)
        .withColumn("stars", (((F.col("stars").cast("int") - 1 + F.floor(F.rand() * 4) + 1) % 5) + 1).cast("double"))
        .withColumn("date", F.expr("least(timestampadd(DAY, 1 + cast(rand() * 30 AS INT), date), current_timestamp())"))
    )

    inserts_df = (
        sampled_df.filter(F.col("_roll") >= UPDATE_RATIO)
        .withColumn("review_id", F.expr("uuid()"))
        .withColumn("user_id", F.expr("uuid()"))
        .withColumn("date", F.current_timestamp())
        .withColumn("stars", F.when(F.rand() < INVALID_RATIO, F.lit(9.0)).otherwise(F.col("stars")))
    )

    batch_dir = f"{landing_incremental_dir('review')}/batch_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"

    (
        updates_df.unionByName(inserts_df)
        .select(*review_columns)
        .coalesce(1)
        .write.mode("error")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .json(batch_dir)
    )
    written_df = spark.read.json(batch_dir)
    written = written_df.count()
    updated = written_df.join(source_df.select("review_id"), "review_id", "left_semi").count()
    invalid = written_df.filter(F.col("stars") > 5).count()

    print(f"batch {batch}/{NUM_BATCHES}: {written} reviews em {batch_dir} ({updated} updates, {written - updated} inserts, {invalid} invalidas)")

    if batch < NUM_BATCHES:
        time.sleep(INTERVAL_SECONDS)